# 🎤 SpeakWise (Claude version) — Streamlit app on Colab

One-session public-speaking coach. Everything (LLM, Whisper, embeddings, MediaPipe) runs on this
Colab runtime; there is no database. Flow:

1. **Random topic** → 2. **~15 min reading material** generated live (Qwen3-4B-Instruct)
3. **Choose 1 / 3 / 5 min** → 4. **Front camera records audio + video** and auto-submits when time is up
5. **Analysis**: filler words, short/long pauses, word repetition, mumbled words, pace, voice
   (faster-whisper large-v3-turbo + librosa); eye contact, posture, gestures, nervous signals
   (MediaPipe Face + Pose landmarkers); context match vs. the material with a **70% pass target**
   (all-mpnet-base-v2 embeddings + LLM judge)
6. **Dashboard** of scores (0-100), counts, an overall summary and improvement tips.

### How to run
1. `Runtime → Change runtime type → T4 GPU` (or better).
2. `Runtime → Run all`. The last cell prints a public **https://…trycloudflare.com** link.
3. Open the link in Chrome/Edge and allow camera + microphone. (HTTPS is required for the camera.)

### Fixes compared with `Speakwise_base.ipynb`
- `audio_path` was never defined (the uploaded MP4's audio was never extracted) → `NameError` on a fresh run. Now ffmpeg extracts a 16 kHz WAV.
- The *Whisper small* cell overwrote the *large-v3-turbo* `asr_model` when running all cells. Only large-v3-turbo is used now.
- Topic mismatch: the evaluator was called with `"AI and technology"` while the material was about `"unicorns"` (and `assigned_topic` was defined *after* use). The topic now comes from the app.
- Models were loaded in the notebook kernel, which a Streamlit process cannot share (and would double GPU memory). The notebook now only downloads models; the app loads them once with `st.cache_resource`.
- No app, topic selector, material generation, duration choice, camera recording, video analysis or dashboard existed (MediaPipe was installed but unused). All added.
- Context matching gave no score; it now returns a 0-100 score with a 70% pass mark.
- Repetition only caught adjacent duplicates; now also overused words and repeated phrases. Added mumbled-word detection, untranscribed "uhh" detection, and pause thresholds that don't count normal 0.25 s word gaps as pauses.
- Contextual fillers (like/so/you know) were listed but never counted; they're now classified with punctuation/pause heuristics and counted.
- Duplicate/unused installs (`streamlit-webrtc`, `av`, `pyngrok`, repeated pip cells) removed.

## 1. Check GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then re-run."
print("GPU:", torch.cuda.get_device_name(0),
      f"({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")

AssertionError: No GPU: Runtime > Change runtime type > T4 GPU, then re-run.

## 2. Install dependencies

In [ ]:
%pip install -q faster-whisper sentence-transformers mediapipe streamlit librosa soundfile
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
!cloudflared --version && ffmpeg -version | head -1

## 3. Write the app

In [ ]:
!mkdir -p /content/speakwise_app/recorder

In [ ]:
%%writefile /content/speakwise_app/content.py
"""Topic selection, reading-material generation and LLM relevance judging.

Runs a local instruction-tuned LLM on the Colab GPU, so no API key is needed.
"""
import json
import os
import random
import re
import threading

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

# Qwen3-4B-Instruct-2507 writes long, well-structured articles and fits a T4 in fp16 (~8 GB).
# Override with the SPEAKWISE_LLM env var (e.g. "Qwen/Qwen2.5-3B-Instruct") if VRAM is tight.
LLM_ID = os.environ.get("SPEAKWISE_LLM", "Qwen/Qwen3-4B-Instruct-2507")

# ~15 minutes of reading at an average adult silent-reading speed of ~220 wpm.
TARGET_WORDS = 3200

TOPICS = [
    "The Future of Remote Work",
    "Artificial Intelligence in Everyday Life",
    "Climate Change and Renewable Energy",
    "The Psychology of Habits",
    "Social Media and Mental Health",
    "Space Exploration and Mars Colonisation",
    "The History and Future of the Internet",
    "Electric Vehicles and Sustainable Transport",
    "The Importance of Sleep",
    "Minimalism and Consumer Culture",
    "Cybersecurity in the Modern World",
    "The Gig Economy",
    "Ocean Plastic Pollution",
    "The Science of Happiness",
    "Online Learning vs Classroom Learning",
    "Urban Farming and Food Security",
    "The Ethics of Genetic Engineering",
    "Cryptocurrency and the Future of Money",
    "The Power of Public Speaking",
    "Video Games as an Art Form",
    "The Role of Sports in Society",
    "Fast Fashion and Its Environmental Cost",
    "Leadership in the 21st Century",
    "The Benefits of Learning a Second Language",
    "Smart Cities",
    "Mythical Creatures in World Cultures",
    "The Rise of Streaming Entertainment",
    "Water Scarcity",
    "Time Management and Productivity",
    "The Human Brain and Memory",
    "Volunteering and Community Service",
    "The Impact of Tourism on Local Cultures",
    "Robots in Healthcare",
    "The Value of Failure",
    "Biodiversity and Endangered Species",
    "Financial Literacy for Young Adults",
    "The Influence of Music on Mood",
    "Self-Driving Cars",
    "The Four-Day Work Week",
    "Misinformation and Fake News",
]


def pick_topic(exclude=None):
    choices = [t for t in TOPICS if t != exclude]
    return random.choice(choices)


def load_llm():
    tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
    # T4 GPUs have no bf16 support, so load in fp16.
    try:
        model = AutoModelForCausalLM.from_pretrained(
            LLM_ID, dtype=torch.float16, device_map="cuda"
        )
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(
            LLM_ID, torch_dtype=torch.float16, device_map="cuda"
        )
    model.eval()
    return tokenizer, model


SYSTEM_PROMPT = (
    "You are an expert educational writer. You write clear, engaging, factual "
    "reading material that prepares a learner to give a short talk on a topic."
)


def _material_prompt(topic):
    return (
        f"Write a comprehensive reading article on the topic: \"{topic}\".\n\n"
        f"Requirements:\n"
        f"- Length: about {TARGET_WORDS} words (roughly 15 minutes of reading).\n"
        "- Use Markdown with a title (#) and 8-10 sections (##).\n"
        "- Cover: background and history, key concepts and definitions, how it works, "
        "real-world examples, benefits, challenges and criticisms, different viewpoints, "
        "recent developments, future outlook, and a conclusion.\n"
        "- End with a section '## Key Talking Points' containing 8 bullet points the "
        "reader could use in a speech.\n"
        "- Write in full paragraphs, plain English, no references or links."
    )


def stream_reading_material(tokenizer, model, topic):
    """Yield text chunks of the article as they are generated.

    Small models often stop early, so if the article is noticeably short we ask the
    model to continue (up to two extra rounds).
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": _material_prompt(topic)},
    ]
    written = ""
    for round_idx in range(3):
        if round_idx > 0:
            if len(written.split()) >= TARGET_WORDS * 0.85:
                break
            messages = messages[:2] + [
                {"role": "assistant", "content": written},
                {
                    "role": "user",
                    "content": (
                        "Continue the article from exactly where it stopped. Add new "
                        "sections with fresh information; do not repeat earlier content "
                        "and do not restart the title. Finish with the Key Talking "
                        "Points section if it is not written yet."
                    ),
                },
            ]
            yield "\n\n"
            written += "\n\n"

        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(model.device)
        streamer = TextIteratorStreamer(
            tokenizer, skip_prompt=True, skip_special_tokens=True, timeout=300
        )
        kwargs = dict(
            **inputs,
            streamer=streamer,
            max_new_tokens=5000,
            do_sample=True,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
            repetition_penalty=1.05,
        )
        thread = threading.Thread(target=model.generate, kwargs=kwargs, daemon=True)
        thread.start()
        for piece in streamer:
            written += piece
            yield piece
        thread.join()


def judge_relevance(tokenizer, model, topic, material, transcript):
    """Ask the LLM how well the speech matches the topic/material.

    Returns {"score": 0-100, "covered_points": [...], "missed_points": [...],
    "off_topic": [...]} or None if the output could not be parsed.
    """
    material_words = material.split()
    if len(material_words) > 3500:
        material = " ".join(material_words[:3500])
    prompt = (
        f"TOPIC: {topic}\n\nREADING MATERIAL:\n{material}\n\n"
        f"SPEECH TRANSCRIPT:\n{transcript}\n\n"
        "The speaker read the material and then gave a short talk. Judge whether the "
        "talk is in context with the topic and the material. The talk does not need to "
        "repeat the material word for word; paraphrases, opinions and relevant examples "
        "count as on-topic. Ignore filler words and grammar.\n\n"
        "Reply with ONLY a JSON object with these keys:\n"
        '{"score": <integer 0-100, how relevant the talk is to the topic/material>, '
        '"covered_points": [<up to 5 short phrases: material ideas the speaker covered>], '
        '"missed_points": [<up to 3 short phrases: important ideas they could add>], '
        '"off_topic": [<up to 3 short phrases: parts of the talk unrelated to the topic>]}'
    )
    messages = [
        {"role": "system", "content": "You are a strict but fair public-speaking coach."},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=400, do_sample=False)
    text = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        data = json.loads(match.group(0))
        score = float(data.get("score"))
    except (ValueError, TypeError):
        return None
    return {
        "score": max(0.0, min(100.0, score)),
        "covered_points": [str(x) for x in data.get("covered_points", [])][:5],
        "missed_points": [str(x) for x in data.get("missed_points", [])][:3],
        "off_topic": [str(x) for x in data.get("off_topic", [])][:3],
    }

In [ ]:
%%writefile /content/speakwise_app/speech.py
"""Speech analysis: transcription, fillers, pauses, repetition, mumbling, pace, voice."""
import math
import re
from collections import Counter

import librosa
import numpy as np
import soundfile as sf
from faster_whisper import WhisperModel

ASR_ID = "large-v3-turbo"

# Whisper tends to "clean up" disfluencies. A prompt written in a disfluent style makes
# it far more likely to keep um/uh/like in the transcript.
DISFLUENT_PROMPT = (
    "Umm, let me think like, hmm... Okay, here's what I'm, like, thinking. "
    "So, uh, I mean, you know, it's- it's kind of, um, like that."
)

FILLERS = {
    "uh": "uh", "uhh": "uh", "uhhh": "uh", "uh-huh": "uh",
    "um": "um", "umm": "um", "ummm": "um", "uhm": "um", "uhmm": "um",
    "erm": "erm", "er": "erm", "err": "erm",
    "ah": "ah", "ahh": "ah", "eh": "eh",
    "hmm": "hmm", "hm": "hmm", "mm": "hmm", "mhm": "hmm", "mmm": "hmm",
}
CONTEXTUAL_FILLERS = {"like", "so", "well", "actually", "basically", "literally",
                      "right", "okay", "ok", "anyway"}
FILLER_PHRASES = {"you know", "i mean", "kind of", "sort of"}

STOPWORDS = set("""
a an the and or but if then so of to in on at by for with from as is are was were be been
being am do does did have has had i you he she it we they me him her us them my your his its
our their this that these those there here what which who whom when where why how not no yes
can could will would shall should may might must just also very really too than about into
over under up down out off again more most some any all each other such only own same both
few many much now then once because while during before after above below between through
like well okay ok right uh um its it's i'm don't that's there's they're we're you're can't
get got go going gonna want say said thing things lot one two
""".split())

SHORT_PAUSE = (0.5, 1.5)   # seconds: natural breath/phrase pause
LONG_PAUSE = 1.5           # seconds and above: noticeable "dead air"


def load_asr():
    return WhisperModel(ASR_ID, device="cuda", compute_type="float16")


def transcribe(asr, wav_path):
    segments, info = asr.transcribe(
        wav_path,
        language="en",
        beam_size=5,
        word_timestamps=True,
        vad_filter=True,
        vad_parameters=dict(threshold=0.35, min_silence_duration_ms=500, speech_pad_ms=300),
        condition_on_previous_text=False,
        initial_prompt=DISFLUENT_PROMPT,
    )
    segments = list(segments)
    words = []
    for seg in segments:
        for w in seg.words or []:
            raw = w.word.strip()
            token = re.sub(r"^[^\w']+|[^\w'-]+$", "", raw.lower()).strip("-")
            if token:
                words.append({
                    "raw": raw,
                    "word": token,
                    "start": float(w.start),
                    "end": float(w.end),
                    "prob": float(w.probability),
                })
    transcript = " ".join(s.text.strip() for s in segments).strip()
    return transcript, words


def _frame_rms(audio, sr, frame_s=0.025):
    hop = int(sr * frame_s)
    rms = librosa.feature.rms(y=audio, frame_length=hop * 2, hop_length=hop)[0]
    flat = librosa.feature.spectral_flatness(y=audio, n_fft=hop * 2, hop_length=hop)[0]
    times = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=hop)
    return times, rms, flat


def _is_contextual_filler(i, words):
    """Heuristic: is this occurrence of like/so/well/... used as a filler?"""
    w = words[i]
    token = w["word"]
    prev = words[i - 1] if i > 0 else None
    nxt = words[i + 1] if i + 1 < len(words) else None
    gap_before = w["start"] - prev["end"] if prev else 1.0
    gap_after = nxt["start"] - w["end"] if nxt else 1.0
    comma_after = w["raw"].endswith(",")
    comma_before = prev is not None and prev["raw"].endswith(",")
    sentence_start = prev is None or prev["raw"][-1:] in ".?!"

    if comma_after or comma_before or gap_before >= 0.4 or gap_after >= 0.4:
        return True
    if token == "like":
        prev_tok = prev["word"] if prev else ""
        next_tok = nxt["word"] if nxt else ""
        return (
            prev_tok in {"and", "but", "so", "um", "uh", "was", "were", "it's", "just", "is"}
            and next_tok not in {"a", "an", "the", "this", "that", "to", "how", "what",
                                 "when", "i", "you", "we", "they", "he", "she", "it"}
        )
    if token in {"so", "well", "okay", "ok", "right", "anyway", "basically", "actually"}:
        return sentence_start
    return False


def analyze_speech(transcript, words, wav_path):
    audio, sr = sf.read(wav_path, dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    duration = len(audio) / sr
    minutes = max(duration / 60.0, 1e-6)

    times, frame_rms, frame_flat = _frame_rms(audio, sr)
    noise_floor = float(np.percentile(frame_rms, 10)) if len(frame_rms) else 0.0

    # ---- per-word loudness ------------------------------------------------
    def seg_rms(a, b):
        seg = audio[int(a * sr): max(int(b * sr), int(a * sr) + 1)]
        return float(np.sqrt(np.mean(seg ** 2))) if len(seg) else 0.0

    for w in words:
        w["rms"] = seg_rms(w["start"], w["end"])
    speech_rms = float(np.median([w["rms"] for w in words])) if words else 0.0

    # ---- fillers ------------------------------------------------------------
    filler_events = []
    for i, w in enumerate(words):
        if w["word"] in FILLERS:
            filler_events.append({"word": FILLERS[w["word"]], "start": w["start"], "kind": "vocal"})
        elif w["word"] in CONTEXTUAL_FILLERS and _is_contextual_filler(i, words):
            filler_events.append({"word": w["word"], "start": w["start"], "kind": "contextual"})
    for i in range(len(words) - 1):
        phrase = f'{words[i]["word"]} {words[i + 1]["word"]}'
        if phrase in FILLER_PHRASES:
            nxt = words[i + 2] if i + 2 < len(words) else None
            punct = words[i + 1]["raw"].endswith(",") or (i > 0 and words[i - 1]["raw"].endswith(","))
            pause = nxt is None or nxt["start"] - words[i + 1]["end"] >= 0.4
            if punct or pause or phrase in {"you know", "i mean"}:
                filler_events.append({"word": phrase, "start": words[i]["start"], "kind": "contextual"})
    filler_word_idx = {i for i, w in enumerate(words) if w["word"] in FILLERS}

    # Whisper still drops some "uhhh"s. Find voiced sound inside gaps between words:
    # loud enough, tonal (low spectral flatness) and >= 0.25 s long.
    hesitations = []
    voiced_thr = max(noise_floor * 3.0, speech_rms * 0.25)
    for i in range(1, len(words)):
        a, b = words[i - 1]["end"], words[i]["start"]
        if b - a < 0.4:
            continue
        mask = (times >= a + 0.05) & (times <= b - 0.05)
        voiced = (frame_rms[mask] > voiced_thr) & (frame_flat[mask] < 0.1)
        run = best = 0
        for v in voiced:
            run = run + 1 if v else 0
            best = max(best, run)
        if best * 0.025 >= 0.25:
            hesitations.append({"start": a, "end": b})
    for h in hesitations:
        filler_events.append({"word": "(uh) untranscribed", "start": h["start"], "kind": "hesitation"})
    filler_events.sort(key=lambda e: e["start"])
    filler_breakdown = Counter(e["word"] for e in filler_events)

    # ---- pauses ---------------------------------------------------------------
    hes_starts = {round(h["start"], 3) for h in hesitations}
    pauses = []
    for i in range(1, len(words)):
        a, b = words[i - 1]["end"], words[i]["start"]
        gap = b - a
        if gap < SHORT_PAUSE[0] or round(a, 3) in hes_starts:
            continue
        pauses.append({
            "start": round(a, 2), "end": round(b, 2), "duration": round(gap, 2),
            "type": "long" if gap >= LONG_PAUSE else "short",
        })
    short_pauses = [p for p in pauses if p["type"] == "short"]
    long_pauses = [p for p in pauses if p["type"] == "long"]
    opening_silence = words[0]["start"] if words else duration

    # ---- repetition -----------------------------------------------------------
    content_tokens = [w["word"] for i, w in enumerate(words) if i not in filler_word_idx]
    stutters = []
    for i in range(1, len(content_tokens)):
        if content_tokens[i] == content_tokens[i - 1] and content_tokens[i] not in {"that", "had", "very"}:
            stutters.append(content_tokens[i])
    for i in range(3, len(content_tokens)):
        if content_tokens[i - 3:i - 1] == content_tokens[i - 1:i + 1]:
            stutters.append(" ".join(content_tokens[i - 1:i + 1]))

    word_count = len(content_tokens)
    meaningful = [t for t in content_tokens if t not in STOPWORDS and len(t) > 2]
    overuse_min = max(3, math.ceil(0.02 * word_count))
    overused = [
        {"word": w, "count": c}
        for w, c in Counter(meaningful).most_common()
        if c >= overuse_min
    ][:8]
    trigrams = Counter(
        tuple(content_tokens[i:i + 3]) for i in range(len(content_tokens) - 2)
        if not all(t in STOPWORDS for t in content_tokens[i:i + 3])
    )
    repeated_phrases = [
        {"phrase": " ".join(k), "count": c} for k, c in trigrams.most_common(10) if c >= 2
    ][:5]

    # ---- mumbling / clarity -----------------------------------------------------
    mumbled = []
    for i, w in enumerate(words):
        if i in filler_word_idx:
            continue
        quiet = speech_rms > 0 and w["rms"] < 0.3 * speech_rms and (w["end"] - w["start"]) > 0.08
        if w["prob"] < 0.45 or quiet:
            mumbled.append({
                "word": w["word"], "start": round(w["start"], 2),
                "reason": "unclear" if w["prob"] < 0.45 else "too quiet",
            })
    clarity_pct = 100.0 * (1 - len(mumbled) / word_count) if word_count else 0.0

    # ---- pace -------------------------------------------------------------------
    span = (words[-1]["end"] - words[0]["start"]) if len(words) > 1 else 0.0
    wpm = word_count / (span / 60.0) if span > 0 else 0.0
    speaking_time = sum(w["end"] - w["start"] for w in words)

    # ---- voice: loudness and pitch variety --------------------------------------
    speech_mask = frame_rms > voiced_thr
    loud_db = 20 * np.log10(frame_rms[speech_mask] + 1e-9) if speech_mask.any() else np.array([-60.0])
    try:
        f0 = librosa.yin(audio, fmin=65, fmax=400, sr=sr, frame_length=1600, hop_length=400)
        f0_rms = librosa.feature.rms(y=audio, frame_length=1600, hop_length=400)[0][: len(f0)]
        f0_flat = librosa.feature.spectral_flatness(y=audio, n_fft=1600, hop_length=400)[0][: len(f0)]
        voiced_f0 = f0[: len(f0_rms)][(f0_rms > voiced_thr) & (f0_flat < 0.1)]
        voiced_f0 = voiced_f0[(voiced_f0 > 70) & (voiced_f0 < 390)]
        if len(voiced_f0) > 20:
            semis = 12 * np.log2(voiced_f0 / np.median(voiced_f0))
            pitch_var = float(np.std(semis[np.abs(semis) < 12]))
        else:
            pitch_var = 0.0
    except Exception:
        pitch_var = 0.0

    # Transcript with fillers / mumbles marked, for the dashboard.
    filler_starts = {round(e["start"], 3) for e in filler_events if e["kind"] != "hesitation"}
    mumble_starts = {m["start"] for m in mumbled}
    marked = []
    for w in words:
        text = w["raw"]
        if round(w["start"], 3) in filler_starts:
            text = f"<mark class='f'>{text}</mark>"
        elif round(w["start"], 2) in mumble_starts:
            text = f"<mark class='m'>{text}</mark>"
        marked.append(text)

    return {
        "transcript": transcript,
        "transcript_marked": " ".join(marked),
        "duration_sec": round(duration, 1),
        "word_count": word_count,
        "speaking_time_sec": round(speaking_time, 1),
        "wpm": round(wpm, 1),
        "opening_silence_sec": round(opening_silence, 1),
        "filler_count": len(filler_events),
        "filler_per_min": round(len(filler_events) / minutes, 2),
        "filler_breakdown": dict(filler_breakdown),
        "filler_events": filler_events,
        "short_pause_count": len(short_pauses),
        "long_pause_count": len(long_pauses),
        "short_pause_total_sec": round(sum(p["duration"] for p in short_pauses), 1),
        "long_pause_total_sec": round(sum(p["duration"] for p in long_pauses), 1),
        "longest_pause_sec": max((p["duration"] for p in pauses), default=0.0),
        "pauses": pauses,
        "stutter_repeats": stutters,
        "stutter_count": len(stutters),
        "overused_words": overused,
        "repeated_phrases": repeated_phrases,
        "mumbled_words": mumbled,
        "mumbled_count": len(mumbled),
        "clarity_pct": round(clarity_pct, 1),
        "loudness_dbfs": round(float(np.median(loud_db)), 1),
        "loudness_variation_db": round(float(np.std(loud_db)), 1),
        "pitch_variation_semitones": round(pitch_var, 2),
    }

In [ ]:
%%writefile /content/speakwise_app/vision.py
"""Video analysis: eye contact, posture, gestures, nervous signals (MediaPipe Tasks API)."""
import math
import os

import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

MODEL_DIR = os.environ.get("SPEAKWISE_MODELS", "/content/speakwise_models")
FACE_TASK = os.path.join(MODEL_DIR, "face_landmarker.task")
POSE_TASK = os.path.join(MODEL_DIR, "pose_landmarker_full.task")

FACE_URL = ("https://storage.googleapis.com/mediapipe-models/face_landmarker/"
            "face_landmarker/float16/1/face_landmarker.task")
POSE_URL = ("https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
            "pose_landmarker_full/float16/1/pose_landmarker_full.task")

SAMPLE_FPS = 10   # face every frame, pose every 2nd frame


def _landmarkers():
    # VIDEO-mode landmarkers are stateful, so create fresh ones for every recording.
    face = mp_vision.FaceLandmarker.create_from_options(mp_vision.FaceLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=FACE_TASK),
        running_mode=mp_vision.RunningMode.VIDEO,
        num_faces=1,
        output_face_blendshapes=True,
    ))
    pose = mp_vision.PoseLandmarker.create_from_options(mp_vision.PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=POSE_TASK),
        running_mode=mp_vision.RunningMode.VIDEO,
        num_poses=1,
    ))
    return face, pose


def _pct(flags):
    return 100.0 * float(np.mean(flags)) if len(flags) else 0.0


def analyze_video(video_path):
    """video_path should already be resampled to SAMPLE_FPS (see pipeline.prepare_media)."""
    face_lm, pose_lm = _landmarkers()
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or SAMPLE_FPS
    frame_ms = 1000.0 / fps

    rows = []          # per-frame face metrics
    pose_rows = []     # per pose frame
    idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        h, w = frame.shape[:2]
        ts = int(idx * frame_ms)
        image = mp.Image(image_format=mp.ImageFormat.SRGB,
                         data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        t = idx / fps

        fres = face_lm.detect_for_video(image, ts)
        if fres.face_landmarks:
            lm = fres.face_landmarks[0]
            bs = {c.category_name: c.score for c in fres.face_blendshapes[0]}
            nose, lcheek, rcheek = lm[1], lm[234], lm[454]
            eye_y = (lm[33].y + lm[263].y) / 2
            chin = lm[152]
            face_w = abs(rcheek.x - lcheek.x) or 1e-6
            yaw_ratio = (nose.x - min(lcheek.x, rcheek.x)) / face_w          # 0.5 = facing camera
            pitch_ratio = (nose.y - eye_y) / max(chin.y - eye_y, 1e-6)      # ~0.45 = level
            facing = abs(yaw_ratio - 0.5) < 0.13 and 0.25 < pitch_ratio < 0.65
            look_h = max(bs.get("eyeLookOutLeft", 0), bs.get("eyeLookOutRight", 0),
                         bs.get("eyeLookInLeft", 0), bs.get("eyeLookInRight", 0))
            look_up = max(bs.get("eyeLookUpLeft", 0), bs.get("eyeLookUpRight", 0))
            look_down = max(bs.get("eyeLookDownLeft", 0), bs.get("eyeLookDownRight", 0))
            gaze_ok = look_h < 0.55 and look_up < 0.55 and look_down < 0.6
            blink = (bs.get("eyeBlinkLeft", 0) + bs.get("eyeBlinkRight", 0)) / 2
            rows.append({
                "t": t, "face": True, "eye_contact": facing and gaze_ok,
                "looking_down": look_down >= 0.6 or pitch_ratio >= 0.65,
                "blink_closed": blink > 0.5,
                "smile": (bs.get("mouthSmileLeft", 0) + bs.get("mouthSmileRight", 0)) / 2 > 0.35,
                "tense": max(bs.get("browDownLeft", 0), bs.get("browDownRight", 0),
                             (bs.get("mouthPressLeft", 0) + bs.get("mouthPressRight", 0)) / 2) > 0.45,
                "brow_raise": bs.get("browInnerUp", 0),
                "nose": (nose.x, nose.y), "face_w": face_w,
            })
        else:
            rows.append({"t": t, "face": False, "eye_contact": False, "looking_down": False,
                         "blink_closed": False, "smile": False, "tense": False,
                         "brow_raise": 0.0, "nose": None, "face_w": None})

        if idx % 2 == 0:
            pres = pose_lm.detect_for_video(image, ts)
            if pres.pose_landmarks:
                p = pres.pose_landmarks[0]
                pt = lambda i: np.array([p[i].x * w, p[i].y * h])
                vis = lambda i: (p[i].visibility or 0) > 0.5
                row = {"t": t, "shoulders": vis(11) and vis(12)}
                if row["shoulders"]:
                    ls, rs, nose = pt(11), pt(12), pt(0)
                    sw = np.linalg.norm(ls - rs) or 1e-6
                    mid = (ls + rs) / 2
                    row["tilt"] = abs(math.degrees(math.atan2(ls[1] - rs[1], ls[0] - rs[0])))
                    row["tilt"] = min(row["tilt"], 180 - row["tilt"])
                    row["neck"] = (mid[1] - nose[1]) / sw
                    row["sw"] = sw
                    row["mid"] = mid
                    wrists = {}
                    for side, wi in (("l", 15), ("r", 16)):
                        if vis(wi):
                            wrists[side] = pt(wi)
                    row["wrists"] = wrists
                    row["touch_face"] = any(
                        np.linalg.norm(wp - nose) < 0.45 * sw for wp in wrists.values()
                    )
                    row["arms_crossed"] = (
                        "l" in wrists and "r" in wrists
                        and (wrists["l"][0] - mid[0]) * (ls[0] - mid[0]) < 0
                        and (wrists["r"][0] - mid[0]) * (rs[0] - mid[0]) < 0
                        and wrists["l"][1] > mid[1] and wrists["r"][1] > mid[1]
                    )
                pose_rows.append(row)
            else:
                pose_rows.append({"t": t, "shoulders": False})
        idx += 1
    cap.release()
    face_lm.close()
    pose_lm.close()

    n = len(rows)
    if n == 0:
        raise RuntimeError("No video frames could be read from the recording.")
    duration = rows[-1]["t"] + 1.0 / fps
    minutes = max(duration / 60.0, 1e-6)
    face_rows = [r for r in rows if r["face"]]

    # Blinks: closed-eye onsets.
    blinks = sum(1 for a, b in zip(rows, rows[1:]) if b["blink_closed"] and not a["blink_closed"])

    # Head movement: nose motion normalised by face width, per second.
    head_speed = []
    for a, b in zip(face_rows, face_rows[1:]):
        dt = b["t"] - a["t"]
        if 0 < dt < 0.5:
            d = math.hypot(b["nose"][0] - a["nose"][0], b["nose"][1] - a["nose"][1])
            head_speed.append(d / b["face_w"] / dt)
    head_speed = np.array(head_speed) if head_speed else np.array([0.0])

    # Posture.
    sp = [r for r in pose_rows if r.get("shoulders")]
    if sp:
        necks = np.array([r["neck"] for r in sp])
        baseline = float(np.percentile(necks, 80))
        slouch = [(r["neck"] < 0.8 * baseline) or (r["neck"] < 0.3) for r in sp]
        tilted = [r["tilt"] > 8 for r in sp]
        crossed = [r["arms_crossed"] for r in sp]
        # Face-touch events (onsets), not frames.
        touches = sum(1 for a, b in zip(sp, sp[1:]) if b["touch_face"] and not a["touch_face"])
        # Hand activity: wrist speed in shoulder-widths per second.
        gest, fidget = [], []
        for a, b in zip(sp, sp[1:]):
            dt = b["t"] - a["t"]
            if not (0 < dt < 0.6):
                continue
            speeds = [np.linalg.norm(b["wrists"][s] - a["wrists"][s]) / b["sw"] / dt
                      for s in ("l", "r") if s in a["wrists"] and s in b["wrists"]]
            if speeds:
                s = max(speeds)
                gest.append(0.25 < s < 2.5)
                fidget.append(s >= 2.5)
        hands_visible = _pct([bool(r["wrists"]) for r in sp])
    else:
        slouch = tilted = crossed = gest = fidget = []
        touches = 0
        hands_visible = 0.0

    # Eye-contact timeline in 5-second bins.
    timeline = []
    bin_s = 5
    for start in range(0, int(math.ceil(duration)), bin_s):
        chunk = [r["eye_contact"] for r in rows if start <= r["t"] < start + bin_s]
        if chunk:
            timeline.append({"second": start, "eye_contact_pct": round(_pct(chunk), 1)})

    brow = np.array([r["brow_raise"] for r in face_rows]) if face_rows else np.array([0.0])

    return {
        "frames_analyzed": n,
        "face_visible_pct": round(_pct([r["face"] for r in rows]), 1),
        "eye_contact_pct": round(_pct([r["eye_contact"] for r in rows]), 1),
        "looking_down_pct": round(_pct([r["looking_down"] for r in rows]), 1),
        "eye_contact_timeline": timeline,
        "blinks_per_min": round(blinks / minutes, 1),
        "smile_pct": round(_pct([r["smile"] for r in face_rows]), 1),
        "tension_pct": round(_pct([r["tense"] for r in face_rows]), 1),
        "expressiveness": round(float(np.std(brow)) * 100, 1),
        "head_movement": round(float(np.median(head_speed)), 3),
        "excess_head_movement_pct": round(_pct(head_speed > 0.6), 1),
        "posture_detected_pct": round(_pct([r.get("shoulders", False) for r in pose_rows]), 1),
        "slouch_pct": round(_pct(slouch), 1),
        "shoulder_tilt_pct": round(_pct(tilted), 1),
        "arms_crossed_pct": round(_pct(crossed), 1),
        "face_touch_count": int(touches),
        "hands_visible_pct": round(hands_visible, 1),
        "gesture_pct": round(_pct(gest), 1),
        "fidget_pct": round(_pct(fidget), 1),
    }

In [ ]:
%%writefile /content/speakwise_app/context.py
"""Context matching: how well the speech matches the topic and the reading material."""
import re

import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_ID = "sentence-transformers/all-mpnet-base-v2"
PASS_SCORE = 70.0

LEADING_FILLERS = re.compile(
    r"^\s*(?:(?:uh+|um+|erm+|er|ah+|hmm+|so|well|like|okay|you know)\b[\s,]*)+", re.I
)


def load_embedder():
    return SentenceTransformer(EMBED_ID, device="cuda")


def _sentences(text):
    text = re.sub(r"(?m)^\s*#{1,6}\s*", "", text)
    text = re.sub(r"(?m)^\s*[-*•]\s*", "", text)
    text = re.sub(r"[*_`]", "", text)
    parts = re.split(r"(?<=[.!?])\s+|\n+", text.strip())
    return [p.strip() for p in parts if len(p.split()) >= 3]


def _windows(sentences, size, step):
    if len(sentences) <= size:
        return [" ".join(sentences)] if sentences else []
    return [" ".join(sentences[i:i + size]) for i in range(0, len(sentences) - size + 1, step)]


def _scale(x, lo, hi):
    return float(np.clip((x - lo) / (hi - lo), 0.0, 1.0))


def context_match(embedder, topic, material, transcript, llm_judgement=None):
    """Return a 0-100 context score (pass >= 70) plus supporting evidence.

    Embedding score = how semantically close each part of the speech is to the closest
    passage in the material, plus whole-speech similarity to topic and material.
    If the LLM judge produced a score, it is blended in (60% embeddings / 40% LLM).
    """
    speech_sents = [LEADING_FILLERS.sub("", s) for s in _sentences(transcript)]
    speech_sents = [s for s in speech_sents if len(s.split()) >= 3]
    words = len(transcript.split())
    if words < 15 or not speech_sents:
        return {"score": 0.0, "passed": False, "embedding_score": 0.0,
                "llm_score": None, "on_topic_pct": 0.0, "material_coverage_pct": 0.0,
                "topic_similarity": 0.0, "sentences": [],
                "note": "Too little speech was detected to judge context."}

    mat_chunks = _windows(_sentences(material), size=3, step=2)
    speech_wins = _windows(speech_sents, size=2, step=1)

    enc = lambda xs: embedder.encode(xs, convert_to_numpy=True, normalize_embeddings=True,
                                     batch_size=32)
    M = enc(mat_chunks)
    S = enc(speech_wins)
    T = enc([topic])[0]
    sims = S @ M.T                                   # (speech windows, material chunks)

    best = sims.max(axis=1)
    per_window = np.array([_scale(b, 0.20, 0.55) for b in best])
    on_topic = best >= 0.40
    topic_sim = float(S.mean(axis=0) @ T / (np.linalg.norm(S.mean(axis=0)) + 1e-9))
    doc_sim = float(
        (S.mean(axis=0) / (np.linalg.norm(S.mean(axis=0)) + 1e-9))
        @ (M.mean(axis=0) / (np.linalg.norm(M.mean(axis=0)) + 1e-9))
    )
    coverage = float((sims.max(axis=0) >= 0.45).mean())

    embedding_score = 100 * (
        0.40 * per_window.mean()
        + 0.30 * on_topic.mean()
        + 0.15 * _scale(doc_sim, 0.25, 0.70)
        + 0.15 * _scale(topic_sim, 0.15, 0.55)
    )
    llm_score = llm_judgement["score"] if llm_judgement else None
    score = embedding_score if llm_score is None else 0.6 * embedding_score + 0.4 * llm_score

    sentence_rows = []
    for i, win in enumerate(speech_wins):
        j = int(sims[i].argmax())
        sentence_rows.append({
            "speech": win,
            "match_pct": round(100 * per_window[i], 1),
            "on_topic": bool(on_topic[i]),
            "closest_material": mat_chunks[j][:220],
        })

    return {
        "score": round(score, 1),
        "passed": score >= PASS_SCORE,
        "embedding_score": round(embedding_score, 1),
        "llm_score": None if llm_score is None else round(llm_score, 1),
        "on_topic_pct": round(100 * on_topic.mean(), 1),
        "material_coverage_pct": round(100 * coverage, 1),
        "topic_similarity": round(topic_sim, 3),
        "sentences": sentence_rows,
        "covered_points": (llm_judgement or {}).get("covered_points", []),
        "missed_points": (llm_judgement or {}).get("missed_points", []),
        "off_topic_points": (llm_judgement or {}).get("off_topic", []),
        "note": "",
    }

In [ ]:
%%writefile /content/speakwise_app/scoring.py
"""Turn raw measurements into 0-100 scores, an overall grade, a summary and tips."""
import numpy as np

WEIGHTS = {
    "fillers": 15, "pauses": 10, "repetition": 10, "clarity": 10, "pace": 10,
    "eye_contact": 15, "body_language": 10, "confidence": 5, "context": 15,
}
LABELS = {
    "fillers": "Filler words", "pauses": "Pauses", "repetition": "Word repetition",
    "clarity": "Clarity (mumbling)", "pace": "Speaking pace", "eye_contact": "Eye contact",
    "body_language": "Body language", "confidence": "Confidence", "context": "Context match",
}


def _clip(x):
    return float(np.clip(x, 0, 100))


def compute_scores(speech, video, context):
    minutes = max(speech["duration_sec"] / 60.0, 1e-6)
    s = {}

    # Fillers: <= 2/min is excellent, 15+/min scores 0.
    s["fillers"] = _clip(100 - (speech["filler_per_min"] - 2) * (100 / 13))

    # Pauses: long pauses hurt; a few short pauses are healthy, too many is choppy.
    long_pm = speech["long_pause_count"] / minutes
    short_pm = speech["short_pause_count"] / minutes
    pause = 100 - 22 * long_pm
    if short_pm > 18:
        pause -= (short_pm - 18) * 3
    if speech["opening_silence_sec"] > 4:
        pause -= 10
    s["pauses"] = _clip(pause)

    # Repetition: stutter-repeats per minute + overused words + repeated phrases.
    s["repetition"] = _clip(
        100 - 12 * speech["stutter_count"] / minutes
        - 6 * len(speech["overused_words"]) - 4 * len(speech["repeated_phrases"])
    )

    s["clarity"] = _clip(100 - (100 - speech["clarity_pct"]) * 4)

    # Pace: 120-160 wpm is ideal.
    wpm = speech["wpm"]
    if wpm == 0:
        s["pace"] = 0.0
    elif wpm < 120:
        s["pace"] = _clip(100 - (120 - wpm) * 1.6)
    elif wpm > 160:
        s["pace"] = _clip(100 - (wpm - 160) * 1.6)
    else:
        s["pace"] = 100.0

    # Eye contact: 75%+ of the time is excellent.
    s["eye_contact"] = _clip(video["eye_contact_pct"] / 75 * 100)

    body = 100.0
    body -= 0.6 * video["slouch_pct"]
    body -= 0.3 * video["shoulder_tilt_pct"]
    body -= 0.5 * video["arms_crossed_pct"]
    body -= 4 * video["face_touch_count"] / minutes
    body -= 0.8 * video["fidget_pct"]
    body -= 0.4 * video["excess_head_movement_pct"]
    body -= 0.5 * max(0.0, 90 - video["face_visible_pct"])
    if video["hands_visible_pct"] > 30 and video["gesture_pct"] < 5:
        body -= 5   # hands visible but completely static
    s["body_language"] = _clip(body)

    # Confidence: blend of visual and vocal signals.
    pitch = speech["pitch_variation_semitones"]
    voice_variety = 100 if 2 <= pitch <= 6 else _clip(40 + pitch * 30) if pitch < 2 else 80
    blink_pen = max(0.0, video["blinks_per_min"] - 30) * 2
    loud = speech["loudness_dbfs"]
    volume = 100 if loud > -30 else _clip(100 - (-30 - loud) * 5)
    s["confidence"] = _clip(
        0.25 * s["eye_contact"] + 0.20 * s["body_language"] + 0.15 * voice_variety
        + 0.15 * volume + 0.15 * s["fillers"] + 0.10 * (100 - video["tension_pct"])
        - blink_pen - 0.15 * video["looking_down_pct"]
    )
    s["voice_variety"] = voice_variety
    s["volume"] = volume

    s["context"] = _clip(context["score"])

    overall = sum(s[k] * w for k, w in WEIGHTS.items()) / sum(WEIGHTS.values())
    s["overall"] = round(overall, 1)
    return {k: round(v, 1) for k, v in s.items()}


def grade(score):
    if score >= 85:
        return "Excellent"
    if score >= 70:
        return "Good"
    if score >= 55:
        return "Fair"
    return "Needs work"


def build_tips(speech, video, context, scores):
    tips = []
    if scores["fillers"] < 80:
        top = sorted(speech["filler_breakdown"].items(), key=lambda kv: -kv[1])[:3]
        names = ", ".join(f'"{w}" ({c})' for w, c in top)
        tips.append((scores["fillers"],
                     f"You used {speech['filler_count']} filler words "
                     f"({speech['filler_per_min']}/min), mostly {names}. Replace them with a "
                     "silent one-second pause - silence sounds more confident than 'um'."))
    if scores["pauses"] < 80:
        tips.append((scores["pauses"],
                     f"{speech['long_pause_count']} long pauses (longest "
                     f"{speech['longest_pause_sec']}s). Plan 3-4 key points before you start "
                     "so you always know what comes next."))
    if scores["repetition"] < 80:
        bits = []
        if speech["overused_words"]:
            bits.append("overused: " + ", ".join(
                f"'{o['word']}' x{o['count']}" for o in speech["overused_words"][:3]))
        if speech["stutter_count"]:
            bits.append(f"{speech['stutter_count']} restarts like 'I I' or 'the the'")
        tips.append((scores["repetition"],
                     "Vary your vocabulary (" + "; ".join(bits) + "). Use synonyms and "
                     "finish each sentence before starting the next."))
    if scores["clarity"] < 80:
        tips.append((scores["clarity"],
                     f"{speech['mumbled_count']} words were unclear or too quiet. Open your "
                     "mouth more, slow down slightly and keep your volume up at sentence ends."))
    if scores["pace"] < 85:
        direction = "slow down" if speech["wpm"] > 160 else "speed up a little"
        tips.append((scores["pace"],
                     f"Your pace was {speech['wpm']:.0f} words/min; aim for 120-160, so "
                     f"{direction}."))
    if scores["eye_contact"] < 80:
        tips.append((scores["eye_contact"],
                     f"Eye contact only {video['eye_contact_pct']:.0f}% of the time"
                     + (f" (looking down {video['looking_down_pct']:.0f}%)"
                        if video["looking_down_pct"] > 15 else "")
                     + ". Look into the camera lens, not at your own image on screen."))
    if scores["body_language"] < 80:
        issues = []
        if video["slouch_pct"] > 20:
            issues.append(f"slouching {video['slouch_pct']:.0f}% of the time")
        if video["shoulder_tilt_pct"] > 20:
            issues.append("leaning to one side")
        if video["face_touch_count"]:
            issues.append(f"touching your face {video['face_touch_count']} times")
        if video["arms_crossed_pct"] > 10:
            issues.append("crossed arms")
        if video["fidget_pct"] > 10:
            issues.append("fidgety hand movements")
        if video["face_visible_pct"] < 90:
            issues.append("moving out of frame")
        tips.append((scores["body_language"],
                     "Body language: " + (", ".join(issues) or "posture could be more open")
                     + ". Sit or stand tall, shoulders level, hands relaxed and visible."))
    if scores["confidence"] < 75:
        extra = []
        if speech["pitch_variation_semitones"] < 2:
            extra.append("your voice was fairly monotone - vary pitch to stress key words")
        if speech["loudness_dbfs"] < -30:
            extra.append("speak louder")
        if video["blinks_per_min"] > 30:
            extra.append("frequent blinking suggests nervousness - slow your breathing")
        tips.append((scores["confidence"],
                     "Project more confidence: " + ("; ".join(extra) or
                     "steady eye contact, open posture and a firm voice") + "."))
    if scores["context"] < 70:
        missed = context.get("missed_points") or []
        tips.append((scores["context"],
                     f"Context match was {context['score']:.0f}% (target 70%). Stay on the "
                     "assigned topic and use facts from the reading"
                     + (": e.g. " + "; ".join(missed[:2]) if missed else "") + "."))
    tips.sort(key=lambda t: t[0])
    if not tips:
        tips.append((100, "Great job! Try a longer duration or a harder topic next time."))
    return [t for _, t in tips]


def build_summary(topic, speech, video, context, scores):
    parts = {k: scores[k] for k in
             ("fillers", "pauses", "repetition", "clarity", "pace",
              "eye_contact", "body_language", "confidence", "context")}
    best = max(parts, key=parts.get)
    worst = min(parts, key=parts.get)
    return (
        f"You spoke about **{topic}** for {speech['duration_sec'] / 60:.1f} min "
        f"({speech['word_count']} words, {speech['wpm']:.0f} wpm) and scored "
        f"**{scores['overall']:.0f}/100 ({grade(scores['overall'])})**. "
        f"Your strongest area was **{LABELS[best].lower()}** ({parts[best]:.0f}) and the "
        f"biggest opportunity is **{LABELS[worst].lower()}** ({parts[worst]:.0f}). "
        f"Your speech was **{context['score']:.0f}% in context** with the topic "
        f"({'passed' if context['passed'] else 'below'} the 70% target), with eye contact "
        f"{video['eye_contact_pct']:.0f}% of the time and {speech['filler_count']} filler words."
    )

In [ ]:
%%writefile /content/speakwise_app/pipeline.py
"""End-to-end analysis of one recording."""
import os
import subprocess

import soundfile as sf

from context import context_match
from content import judge_relevance
from scoring import build_summary, build_tips, compute_scores
from speech import analyze_speech, transcribe
from vision import SAMPLE_FPS, analyze_video


def prepare_media(src_path, workdir):
    """Browser recordings are WebM with variable frame rate and no duration header.
    Normalise to a 16 kHz mono WAV (for ASR) and a constant-fps MP4 (for MediaPipe)."""
    os.makedirs(workdir, exist_ok=True)
    wav = os.path.join(workdir, "audio.wav")
    mp4 = os.path.join(workdir, "video.mp4")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src_path,
                    "-vn", "-ac", "1", "-ar", "16000", wav], check=True)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src_path, "-an",
                    "-vf", f"fps={SAMPLE_FPS},scale=640:-2", "-c:v", "libx264",
                    "-preset", "ultrafast", mp4], check=True)
    return wav, mp4, sf.info(wav).duration


def run_analysis(src_path, topic, material, asr, embedder, llm=None, progress=None):
    step = progress or (lambda msg: None)
    workdir = os.path.join(os.path.dirname(src_path), "work")

    step("Extracting audio and video tracks...")
    wav, mp4, _ = prepare_media(src_path, workdir)

    step("Transcribing speech (Whisper large-v3-turbo)...")
    transcript, words = transcribe(asr, wav)

    step("Measuring fillers, pauses, repetition and clarity...")
    speech = analyze_speech(transcript, words, wav)

    step("Analysing eye contact and body language (MediaPipe)...")
    video = analyze_video(mp4)

    step("Checking context against the reading material...")
    judgement = None
    if llm is not None and len(transcript.split()) >= 15:
        try:
            judgement = judge_relevance(llm[0], llm[1], topic, material, transcript)
        except Exception as exc:  # the embedding score still works without the judge
            print("LLM judge failed:", exc)
    context = context_match(embedder, topic, material, transcript, judgement)

    step("Scoring...")
    scores = compute_scores(speech, video, context)
    return {
        "topic": topic,
        "speech": speech,
        "video": video,
        "context": context,
        "scores": scores,
        "tips": build_tips(speech, video, context, scores),
        "summary": build_summary(topic, speech, video, context, scores),
    }

In [ ]:
%%writefile /content/speakwise_app/app.py
"""SpeakWise - one-session public-speaking coach (Streamlit, runs on a Colab GPU)."""
import base64
import json
import os
import tempfile

import pandas as pd
import streamlit as st
import streamlit.components.v1 as components

from content import load_llm, pick_topic, stream_reading_material
from context import PASS_SCORE, load_embedder
from pipeline import run_analysis
from scoring import LABELS, WEIGHTS, grade
from speech import load_asr

st.set_page_config(page_title="SpeakWise", page_icon="🎤", layout="wide")
st.markdown("""
<style>
mark.f { background: #ffb4a8; color: #000; border-radius: 3px; padding: 0 2px; }
mark.m { background: #ffe08a; color: #000; border-radius: 3px; padding: 0 2px; }
.big-score { font-size: 64px; font-weight: 800; line-height: 1; }
</style>
""", unsafe_allow_html=True)

APP_DIR = os.path.dirname(os.path.abspath(__file__))
recorder = components.declare_component("recorder", path=os.path.join(APP_DIR, "recorder"))


@st.cache_resource(show_spinner=False)
def get_llm():
    return load_llm()


@st.cache_resource(show_spinner=False)
def get_asr():
    return load_asr()


@st.cache_resource(show_spinner=False)
def get_embedder():
    return load_embedder()


def warm_up():
    with st.spinner("Loading AI models onto the GPU (first launch only, ~1-2 min)..."):
        get_llm(), get_asr(), get_embedder()


ss = st.session_state
ss.setdefault("stage", "home")


def go(stage):
    ss.stage = stage
    st.rerun()


def reset():
    for k in list(ss.keys()):
        del ss[k]
    st.rerun()


# ---------------------------------------------------------------- 1. home
if ss.stage == "home":
    st.title("🎤 SpeakWise")
    st.subheader("Read. Speak. Get scored.")
    st.write(
        "1. Get a random topic  2. Read ~15 minutes of material  3. Choose 1, 3 or 5 minutes "
        "4. Speak to your camera  5. Get a full analysis of your delivery, body language and content."
    )
    warm_up()
    if st.button("🎲 Pick a random topic", type="primary", width="stretch"):
        ss.topic = pick_topic()
        ss.material = None
        go("reading")

# ---------------------------------------------------------------- 2-3. reading
elif ss.stage == "reading":
    st.caption("YOUR TOPIC")
    st.title(ss.topic)
    if ss.material is None:
        st.info("Your reading material is being written live - start reading as it appears.")
        tok, model = get_llm()
        ss.material = st.write_stream(stream_reading_material(tok, model, ss.topic))
        st.rerun()
    words = len(ss.material.split())
    st.caption(f"{words:,} words · about {max(1, round(words / 220))} min read")
    st.markdown(ss.material)
    st.divider()
    c1, c2 = st.columns([3, 1])
    if c1.button("✅ I've finished reading", type="primary", width="stretch"):
        go("duration")
    if c2.button("🎲 Different topic", width="stretch"):
        ss.topic = pick_topic(exclude=ss.topic)
        ss.material = None
        st.rerun()

# ---------------------------------------------------------------- 4. duration
elif ss.stage == "duration":
    st.title("How long do you want to speak?")
    st.write(f"Topic: **{ss.topic}**. Your camera opens and recording starts 3 seconds "
             "after you choose. It stops and submits automatically when time is up.")
    cols = st.columns(3)
    for col, mins in zip(cols, (1, 3, 5)):
        if col.button(f"{mins} minute{'s' if mins > 1 else ''}", width="stretch",
                      type="primary"):
            ss.duration = mins * 60
            ss.rec_nonce = ss.get("rec_nonce", 0) + 1
            go("record")

# ---------------------------------------------------------------- 5. record
elif ss.stage == "record":
    st.title(f"Speak about: {ss.topic}")
    st.caption("Look at the camera lens, sit up straight and speak clearly.")
    value = recorder(duration=ss.duration, key=f"rec_{ss.rec_nonce}", default=None)
    with st.expander("Camera not working? Upload a recording instead"):
        up = st.file_uploader("Video file", type=["mp4", "webm", "mov", "mkv"])
    data, suffix = None, ".webm"
    if value and value.get("b64"):
        data = base64.b64decode(value["b64"])
        suffix = ".mp4" if "mp4" in value.get("mime", "") else ".webm"
    elif up is not None:
        data, suffix = up.getvalue(), os.path.splitext(up.name)[1] or ".mp4"
    if data:
        folder = tempfile.mkdtemp(prefix="speakwise_")
        ss.video_path = os.path.join(folder, "recording" + suffix)
        with open(ss.video_path, "wb") as f:
            f.write(data)
        go("analyzing")

# ---------------------------------------------------------------- 6-7. analysis
elif ss.stage == "analyzing":
    st.title("Analysing your speech...")
    with st.status("Running analysis", expanded=True) as status:
        try:
            ss.results = run_analysis(
                ss.video_path, ss.topic, ss.material,
                asr=get_asr(), embedder=get_embedder(), llm=get_llm(),
                progress=st.write,
            )
        except Exception as exc:
            status.update(label="Analysis failed", state="error")
            st.exception(exc)
            if st.button("Try recording again"):
                go("duration")
            st.stop()
        status.update(label="Done!", state="complete")
    go("dashboard")

# ---------------------------------------------------------------- 8. dashboard
elif ss.stage == "dashboard":
    r = ss.results
    sp, vd, cx, sc = r["speech"], r["video"], r["context"], r["scores"]

    st.title("📊 Your SpeakWise Report")
    st.caption(f"Topic: {r['topic']}")
    left, right = st.columns([1, 3])
    with left:
        st.markdown(f"<div class='big-score'>{sc['overall']:.0f}</div>", unsafe_allow_html=True)
        st.markdown(f"**/100 · {grade(sc['overall'])}**")
    with right:
        st.markdown(r["summary"])

    st.subheader("Scores")
    keys = list(WEIGHTS)
    for row in (keys[:5], keys[5:]):
        cols = st.columns(len(row))
        for col, k in zip(cols, row):
            col.metric(LABELS[k], f"{sc[k]:.0f}/100")
            col.progress(int(sc[k]) / 100)

    st.subheader("🗣️ Speech delivery")
    c = st.columns(6)
    c[0].metric("Filler words", sp["filler_count"], f"{sp['filler_per_min']}/min",
                delta_color="off")
    c[1].metric("Long pauses (≥1.5s)", sp["long_pause_count"],
                f"{sp['long_pause_total_sec']}s total", delta_color="off")
    c[2].metric("Short pauses (0.5-1.5s)", sp["short_pause_count"],
                f"{sp['short_pause_total_sec']}s total", delta_color="off")
    c[3].metric("Repeated words", sp["stutter_count"] + sum(o["count"] for o in sp["overused_words"]),
                f"{sp['stutter_count']} restarts", delta_color="off")
    c[4].metric("Mumbled words", sp["mumbled_count"], f"{sp['clarity_pct']}% clear",
                delta_color="off")
    c[5].metric("Pace", f"{sp['wpm']:.0f} wpm", "ideal 120-160", delta_color="off")

    d1, d2 = st.columns(2)
    with d1:
        st.markdown("**Filler words used**")
        if sp["filler_breakdown"]:
            st.bar_chart(pd.Series(sp["filler_breakdown"], name="count").sort_values(ascending=False),
                         horizontal=True)
        else:
            st.success("No filler words detected!")
    with d2:
        st.markdown("**Most repeated words & phrases**")
        rep = [{"word / phrase": o["word"], "times": o["count"]} for o in sp["overused_words"]]
        rep += [{"word / phrase": p["phrase"], "times": p["count"]} for p in sp["repeated_phrases"]]
        if sp["stutter_repeats"]:
            rep.append({"word / phrase": "restarts: " + ", ".join(sp["stutter_repeats"][:6]),
                        "times": sp["stutter_count"]})
        if rep:
            st.dataframe(pd.DataFrame(rep), hide_index=True, width="stretch")
        else:
            st.success("No noticeable repetition.")
    v = st.columns(3)
    v[0].metric("Words spoken", sp["word_count"])
    v[1].metric("Pitch variation", f"{sp['pitch_variation_semitones']} st",
                "monotone" if sp["pitch_variation_semitones"] < 2 else "lively", delta_color="off")
    v[2].metric("Volume", f"{sp['loudness_dbfs']} dBFS",
                "too quiet" if sp["loudness_dbfs"] < -30 else "good", delta_color="off")

    st.subheader("🧍 Body language & eye contact")
    b = st.columns(6)
    b[0].metric("Eye contact", f"{vd['eye_contact_pct']:.0f}%")
    b[1].metric("Looking down", f"{vd['looking_down_pct']:.0f}%")
    b[2].metric("Slouching", f"{vd['slouch_pct']:.0f}%")
    b[3].metric("Face touches", vd["face_touch_count"])
    b[4].metric("Gesturing", f"{vd['gesture_pct']:.0f}%")
    b[5].metric("Blinks / min", vd["blinks_per_min"])
    b2 = st.columns(6)
    b2[0].metric("Face in frame", f"{vd['face_visible_pct']:.0f}%")
    b2[1].metric("Shoulder tilt", f"{vd['shoulder_tilt_pct']:.0f}%")
    b2[2].metric("Arms crossed", f"{vd['arms_crossed_pct']:.0f}%")
    b2[3].metric("Fidgeting", f"{vd['fidget_pct']:.0f}%")
    b2[4].metric("Smiling", f"{vd['smile_pct']:.0f}%")
    b2[5].metric("Facial tension", f"{vd['tension_pct']:.0f}%")
    if vd["eye_contact_timeline"]:
        st.markdown("**Eye contact over time (% per 5 s)**")
        tl = pd.DataFrame(vd["eye_contact_timeline"]).set_index("second")
        st.line_chart(tl["eye_contact_pct"], y_label="eye contact %", x_label="seconds")

    st.subheader("🎯 Context match with the topic")
    k = st.columns(4)
    k[0].metric("Context score", f"{cx['score']:.0f}%",
                "✅ on topic" if cx["passed"] else f"❌ below {PASS_SCORE:.0f}% target",
                delta_color="off")
    k[1].metric("Speech on-topic", f"{cx['on_topic_pct']:.0f}%")
    k[2].metric("Material covered", f"{cx['material_coverage_pct']:.0f}%")
    k[3].metric("AI judge score", "-" if cx["llm_score"] is None else f"{cx['llm_score']:.0f}%")
    if cx.get("note"):
        st.warning(cx["note"])
    p1, p2, p3 = st.columns(3)
    for col, title, items in ((p1, "Covered well", cx.get("covered_points")),
                              (p2, "Could have mentioned", cx.get("missed_points")),
                              (p3, "Off-topic parts", cx.get("off_topic_points"))):
        col.markdown(f"**{title}**")
        col.markdown("\n".join(f"- {x}" for x in items) if items else "_none_")
    with st.expander("Sentence-by-sentence match"):
        if cx["sentences"]:
            st.dataframe(pd.DataFrame(cx["sentences"]), hide_index=True, width="stretch")

    st.subheader("💡 Improvement tips")
    for i, tip in enumerate(r["tips"], 1):
        st.markdown(f"**{i}.** {tip}")

    with st.expander("Transcript (red = filler, yellow = mumbled/unclear)"):
        st.markdown(sp["transcript_marked"] or "_No speech detected._", unsafe_allow_html=True)
    with st.expander("Watch your recording"):
        st.video(ss.video_path)

    st.divider()
    c1, c2 = st.columns(2)
    c1.download_button("⬇️ Download full report (JSON)",
                       json.dumps(r, indent=2, default=str), "speakwise_report.json",
                       width="stretch")
    if c2.button("🔁 Start a new session", type="primary", width="stretch"):
        reset()

In [ ]:
%%writefile /content/speakwise_app/cli.py
"""Headless test: python cli.py VIDEO --topic "..." [--material file.txt] [--no-llm]

Runs the same analysis as the app (in a separate process, so GPU memory is freed when
it exits) and prints the scores. Without --material, reading material is generated.
"""
import argparse
import json

from content import load_llm, stream_reading_material
from context import load_embedder
from pipeline import run_analysis
from speech import load_asr

ap = argparse.ArgumentParser()
ap.add_argument("video")
ap.add_argument("--topic", required=True)
ap.add_argument("--material")
ap.add_argument("--no-llm", action="store_true")
args = ap.parse_args()

llm = None if args.no_llm else load_llm()
if args.material:
    material = open(args.material, encoding="utf-8-sig").read()
elif llm:
    print("Generating reading material...")
    material = "".join(stream_reading_material(llm[0], llm[1], args.topic))
else:
    raise SystemExit("Pass --material when using --no-llm")

result = run_analysis(args.video, args.topic, material, load_asr(), load_embedder(), llm,
                      progress=print)
print("\nSUMMARY:", result["summary"])
print("\nSCORES:", json.dumps(result["scores"], indent=2))
print("\nTIPS:")
for tip in result["tips"]:
    print(" -", tip)
with open("/content/speakwise_test_report.json", "w") as f:
    json.dump(result, f, indent=2, default=str)
print("\nFull report saved to /content/speakwise_test_report.json")

In [ ]:
%%writefile /content/speakwise_app/recorder/index.html
<!doctype html>
<html>
<head>
<meta charset="utf-8">
<style>
  body { margin: 0; font-family: "Source Sans Pro", sans-serif; color: #e8e8e8; background: transparent; }
  #wrap { max-width: 720px; margin: 0 auto; }
  #stage { position: relative; background: #000; border-radius: 12px; overflow: hidden; aspect-ratio: 4 / 3; }
  video { width: 100%; height: 100%; object-fit: cover; transform: scaleX(-1); display: block; }
  #count { position: absolute; inset: 0; display: flex; align-items: center; justify-content: center;
           font-size: 120px; font-weight: 700; color: #fff; text-shadow: 0 4px 20px #000; }
  #rec { position: absolute; top: 12px; left: 12px; display: none; align-items: center; gap: 8px;
         background: rgba(0,0,0,.6); padding: 6px 12px; border-radius: 20px; font-weight: 600; }
  #rec i { width: 12px; height: 12px; border-radius: 50%; background: #ff3b3b; animation: blink 1s infinite; }
  @keyframes blink { 50% { opacity: .2; } }
  #bar { height: 8px; background: #333; border-radius: 4px; margin-top: 12px; overflow: hidden; }
  #fill { height: 100%; width: 0; background: linear-gradient(90deg, #4f8bf9, #7c4dff); transition: width .5s linear; }
  #row { display: flex; justify-content: space-between; align-items: center; margin-top: 10px; }
  #status { font-size: 16px; color: #bbb; }
  button { background: #ff4b4b; color: #fff; border: 0; padding: 10px 18px; border-radius: 8px;
           font-size: 15px; cursor: pointer; display: none; }
</style>
</head>
<body>
<div id="wrap">
  <div id="stage">
    <video id="preview" autoplay muted playsinline></video>
    <div id="count"></div>
    <div id="rec"><i></i><span id="timer">0:00</span></div>
  </div>
  <div id="bar"><div id="fill"></div></div>
  <div id="row">
    <span id="status">Allow camera and microphone access to begin.</span>
    <button id="stop">Finish early &amp; submit</button>
  </div>
</div>
<script>
  const send = (type, data) =>
    window.parent.postMessage(Object.assign({ isStreamlitMessage: true, type }, data), "*");
  const $ = (id) => document.getElementById(id);
  const fmt = (s) => `${Math.floor(s / 60)}:${String(Math.floor(s % 60)).padStart(2, "0")}`;
  const setHeight = () => send("streamlit:setFrameHeight", { height: document.body.scrollHeight + 10 });

  let duration = 60, initialised = false, stream, recorder, chunks = [], t0, ticker;

  window.addEventListener("message", (ev) => {
    if (!ev.data || ev.data.type !== "streamlit:render") return;
    duration = (ev.data.args && ev.data.args.duration) || 60;
    setHeight();
    if (!initialised) { initialised = true; start(); }
  });
  window.addEventListener("resize", setHeight);
  send("streamlit:componentReady", { apiVersion: 1 });

  async function start() {
    try {
      stream = await navigator.mediaDevices.getUserMedia({
        video: { facingMode: "user", width: { ideal: 640 }, height: { ideal: 480 }, frameRate: { ideal: 15, max: 24 } },
        audio: { echoCancellation: true, noiseSuppression: true, autoGainControl: true },
      });
    } catch (err) {
      $("status").textContent = "Camera/microphone blocked: " + err.message +
        ". Allow access in the browser address bar and reload, or use the upload option below.";
      setHeight();
      return;
    }
    $("preview").srcObject = stream;
    $("status").textContent = "Get ready - recording starts automatically.";
    for (let n = 3; n > 0; n--) { $("count").textContent = n; await new Promise((r) => setTimeout(r, 1000)); }
    $("count").textContent = "";
    record();
  }

  function record() {
    const types = ["video/webm;codecs=vp8,opus", "video/webm;codecs=vp9,opus", "video/webm", "video/mp4"];
    const mimeType = types.find((t) => window.MediaRecorder && MediaRecorder.isTypeSupported(t)) || "";
    recorder = new MediaRecorder(stream, { mimeType, videoBitsPerSecond: 500000, audioBitsPerSecond: 96000 });
    recorder.ondataavailable = (e) => e.data.size && chunks.push(e.data);
    recorder.onstop = submit;
    recorder.start(1000);
    t0 = Date.now();
    $("rec").style.display = "flex";
    $("stop").style.display = "inline-block";
    $("status").textContent = "Recording... speak now!";
    ticker = setInterval(() => {
      const el = (Date.now() - t0) / 1000;
      $("timer").textContent = fmt(el) + " / " + fmt(duration);
      $("fill").style.width = Math.min(100, (100 * el) / duration) + "%";
      if (el >= duration) stop();
    }, 250);
    $("stop").onclick = stop;
  }

  function stop() {
    if (!recorder || recorder.state === "inactive") return;
    clearInterval(ticker);
    recorder.stop();
    stream.getTracks().forEach((t) => t.stop());
    $("rec").style.display = "none";
    $("stop").style.display = "none";
    $("status").textContent = "Time's up! Uploading your recording...";
  }

  function submit() {
    const blob = new Blob(chunks, { type: recorder.mimeType || "video/webm" });
    const reader = new FileReader();
    reader.onloadend = () => {
      const b64 = reader.result.split(",")[1];
      send("streamlit:setComponentValue", {
        value: { b64, mime: blob.type, seconds: (Date.now() - t0) / 1000 },
        dataType: "json",
      });
      $("status").textContent = "Submitted - analysing...";
    };
    reader.readAsDataURL(blob);
  }
</script>
</body>
</html>

## 4. Download models (cached on disk; loaded by the app, not here)

In [ ]:
import os, sys, urllib.request
sys.path.insert(0, "/content/speakwise_app")
from huggingface_hub import snapshot_download
from faster_whisper.utils import download_model
from content import LLM_ID
from context import EMBED_ID
from vision import MODEL_DIR, FACE_TASK, FACE_URL, POSE_TASK, POSE_URL

print("LLM:", snapshot_download(LLM_ID))
print("Embedder:", snapshot_download(EMBED_ID))
print("Whisper:", download_model("large-v3-turbo"))
os.makedirs(MODEL_DIR, exist_ok=True)
for path, url in ((FACE_TASK, FACE_URL), (POSE_TASK, POSE_URL)):
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    print("MediaPipe:", path, os.path.getsize(path) // 1024, "KB")

## 5. (Optional) Headless test on an uploaded video
Runs the full analysis in a separate process (GPU memory is freed afterwards). Skip this when you just want the app.

In [ ]:
RUN_TEST = False   # set True to test with an uploaded video
if RUN_TEST:
    from google.colab import files
    up = files.upload()
    video = "/content/" + next(iter(up))
    with open(video, "wb") as f:
        f.write(up[next(iter(up))])
    !cd /content/speakwise_app && python cli.py "{video}" --topic "Mythical Creatures in World Cultures"

## 6. Launch the app

In [ ]:
import re, subprocess, sys, time, urllib.request

subprocess.run("pkill -f 'streamlit run'; pkill -f 'cloudflared tunnel'", shell=True)
st_log = open("/content/streamlit.log", "w")
st_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port", "8501", "--server.headless", "true",
     "--server.maxMessageSize", "600", "--server.maxUploadSize", "600",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
     "--browser.gatherUsageStats", "false"],
    cwd="/content/speakwise_app", stdout=st_log, stderr=subprocess.STDOUT)

for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Streamlit did not start:\n" + open("/content/streamlit.log").read())

cf_log = open("/content/cloudflared.log", "w")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8501"],
    stdout=cf_log, stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if m:
        url = m.group(0)
        break
    time.sleep(1)
print("✅ SpeakWise is running at:", url or "(no URL yet - check /content/cloudflared.log)")
print("Models load on the first visit (~1-2 min). Keep this Colab tab open while using the app.")

## 7. Logs / stop

In [ ]:
!tail -n 40 /content/streamlit.log

In [ ]:
# Stop the app and tunnel
!pkill -f 'streamlit run'; pkill -f 'cloudflared tunnel'; echo stopped